# Movie Recommendation Engine — Learning Walkthrough

Run cells in order. Each section matches a script in `steps/`.

**Concepts:** feature engineering → cosine similarity → nearest neighbors → recommendations

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from scripts.download_data import download_movielens
from src.data_loader import load_all
from src.explore import explore_dataset, print_summary

download_movielens()
data = load_all()
summary = explore_dataset(data["movies"], data["ratings"], data["tags"])
print_summary(summary)

## Step 2: Feature Engineering

Build a **user-item rating matrix** and **genre one-hot** features.

In [ ]:
from src.features import (
    build_movie_genre_features,
    build_user_item_matrix,
    filter_active_users_and_movies,
)

ratings = filter_active_users_and_movies(data["ratings"])
user_item_df, _ = build_user_item_matrix(ratings)
genre_df, genre_np = build_movie_genre_features(data["movies"])

print(f"User-item matrix: {user_item_df.shape}")
print(f"Genre features: {genre_df.shape}")
user_item_df.iloc[:3, :5]

## Step 3: Cosine Similarity

`similarity(a,b) = dot(a,b) / (norm(a) * norm(b))`

In [ ]:
from src.similarity import cosine_similarity_manual, cosine_similarity_matrix

a = genre_np[0]
b = genre_np[1]
print(f"Manual cosine sim: {cosine_similarity_manual(a, b):.4f}")

sim = cosine_similarity_matrix(genre_np[:100])
print(f"Similarity matrix shape: {sim.shape}")

## Step 5: Recommendations

Train three recommenders and compare outputs for one user.

In [ ]:
from src.recommender import ItemBasedCF, UserBasedCF, ContentBasedRecommender, format_recommendations

movies, ratings, tags = data["movies"], data["ratings"], data["tags"]
demo_user = ratings.groupby("userId").size().idxmax()

item_cf = ItemBasedCF().fit(movies, ratings)
user_cf = UserBasedCF().fit(movies, ratings)
content = ContentBasedRecommender(use_tags=True).fit(movies, tags)

print("Item-Based CF:")
print(format_recommendations(item_cf.recommend_for_user(demo_user, top_k=5)))
print("\nUser-Based CF:")
print(format_recommendations(user_cf.recommend_for_user(demo_user, top_k=5)))
print("\nContent-Based:")
print(format_recommendations(content.recommend_for_user(ratings, demo_user, top_k=5)))